# 01 · Clean TSAC

TSAC cleaning funnel using shared loaders/helpers and fixed output paths.

In [1]:
# Works both locally (repo root or notebooks/) and in Colab after cloning the repo.
import os, sys
from pathlib import Path

if Path("notebooks").is_dir():
    ROOT = Path(".").resolve()
elif Path("../notebooks").is_dir():
    ROOT = Path("..").resolve()
else:  # Colab: clone first, e.g. !git clone https://github.com/<owner>/tunisian-sentiment.git
    ROOT = Path("/content/tunisian-sentiment")

sys.path.insert(0, str(ROOT))
os.chdir(ROOT)
print("Repo root:", ROOT)


Repo root: /home/runner/work/Data_Project/Data_Project/tunisian-sentiment/tunisian-sentiment


In [2]:
import pandas as pd
from src.cleaning_utils import fix_text, has_letter, CleaningLog
from src.load_raw import load_tsac

OUT = ROOT / "data/interim/tsac_clean.csv"
LOG = ROOT / "logs/cleaning_log_tsac.csv"
log = CleaningLog()


In [3]:
# Load train and test separately (ids/split are assigned by the shared loader).
train = load_tsac("train", ROOT)
test = load_tsac("test", ROOT)

print("Raw train rows:", len(train))
print("Raw test rows:", len(test))
assert len(train) == 13669, f"Unexpected raw train size: {len(train)}"
assert len(test) == 3400, f"Unexpected raw test size: {len(test)}"

df = pd.concat([train, test], ignore_index=True)
log.add("load", len(df), len(df), f"TSAC train {len(train)}, test {len(test)}")

df.head()


Raw train rows: 13669
Raw test rows: 3400
load                                            17069 ->   17069  (removed 0)


,id,text,label,split
0,tsac_train_00001,Oui ene t3jbni barcha,pos,train
1,tsac_train_00002,تبارك الله عليه,pos,train
2,tsac_train_00003,امرأة أنيقة و سلسة في خطابها و ذكية و متواضعة,pos,train
3,tsac_train_00004,جعفر يحب بلادو ويعمل في الخير يعاون في الناس و...,pos,train
4,tsac_train_00005,Mahleha,pos,train


In [4]:
print("Label values:", sorted(df["label"].dropna().unique().tolist()))
print(df["label"].value_counts())
assert set(df["label"].dropna().unique()) <= {"pos", "neg"}, "Unexpected labels found"


Label values: ['neg', 'pos']
label
pos    8854
neg    8215
Name: count, dtype: int64


In [5]:
# Normalize _x000D_ artifacts + whitespace only (no lowercasing/transliteration/rewriting).
n_x000d = df["text"].str.contains("_x000D_", regex=False).sum()
changed = (df["text"] != df["text"].map(fix_text)).sum()

df["text"] = df["text"].map(fix_text)

log.add(
    "fix_text (_x000D_, spaces)",
    len(df),
    len(df),
    f"{n_x000d} rows had _x000D_; {changed} rows changed"
)


fix_text (_x000D_, spaces)                      17069 ->   17069  (removed 0)


In [6]:
before = len(df)
df = df[df["text"] != ""].copy()
log.add("remove empty rows", before, len(df))

before = len(df)
df = df[df["text"].map(has_letter)].copy()
log.add("remove rows without any letter", before, len(df))


remove empty rows                               17069 ->   17061  (removed 8)
remove rows without any letter                  17061 ->   16977  (removed 84)


In [7]:
# Proposal reference mentions 1,791 duplicate-text references; compute and report actual value.
duplicate_text_references = int(df.duplicated(subset=["text"]).sum())
print(f"Duplicate-text references before dedup: {duplicate_text_references} (proposal reference: 1791)")


Duplicate-text references before dedup: 1780 (proposal reference: 1791)


In [8]:
# Label conflicts: same text with both pos/neg anywhere in TSAC -> remove all copies.
labels_per_text = df.groupby("text")["label"].nunique()
conflict_texts = set(labels_per_text[labels_per_text > 1].index)

before = len(df)
df = df[~df["text"].isin(conflict_texts)].copy()
removed_conflict_rows = before - len(df)
removed_conflict_texts = len(conflict_texts)

log.add(
    "remove label conflicts (all copies)",
    before,
    len(df),
    f"{removed_conflict_texts} conflicting texts; removed {removed_conflict_rows} rows"
)
print(f"Conflicting texts removed: {removed_conflict_texts} (brief expected ~14)")


remove label conflicts (all copies)             16977 ->   16873  (removed 104)
Conflicting texts removed: 12 (brief expected ~14)


In [9]:
# Duplicates within each split: keep one copy.
before = len(df)
within_split_dup_mask = df.duplicated(subset=["split", "text"], keep="first")
within_split_duplicates_removed = int(within_split_dup_mask.sum())
df = df[~within_split_dup_mask].copy()

log.add(
    "remove duplicates within split (keep first)",
    before,
    len(df),
    f"removed {within_split_duplicates_removed} duplicate rows within split"
)


remove duplicates within split (keep first)     16873 ->   15498  (removed 1375)


In [10]:
# Train/test leakage: if text appears in both splits, keep test copy and remove train copy only.
train_texts = set(df.loc[df["split"] == "train", "text"])
test_texts = set(df.loc[df["split"] == "test", "text"])
leak_texts = train_texts & test_texts

before = len(df)
train_before = int((df["split"] == "train").sum())

df = df[~((df["split"] == "train") & (df["text"].isin(leak_texts)))].copy()

train_rows_removed_due_to_leakage = train_before - int((df["split"] == "train").sum())
leakage_text_count = len(leak_texts)

log.add(
    "remove train/test leakage (drop train copy)",
    before,
    len(df),
    f"{leakage_text_count} leaked texts; removed {train_rows_removed_due_to_leakage} train rows"
)
print(f"Leakage texts: {leakage_text_count} (brief expected ~319)")


remove train/test leakage (drop train copy)     15498 ->   15185  (removed 313)
Leakage texts: 313 (brief expected ~319)


In [11]:
df["source"] = "tsac"
df = df[["id", "text", "label", "source", "split"]]

# Required checks before saving.
assert df["text"].notna().all(), "Missing text values remain"
assert (df["text"] != "").all(), "Empty text remains"
assert set(df["label"].dropna().unique()) <= {"pos", "neg"}, "Unexpected labels remain"
assert df["id"].is_unique, "IDs are not unique"
assert not df["text"].duplicated().any(), "Duplicate text remains"
assert len(set(df.loc[df["split"] == "train", "text"]) & set(df.loc[df["split"] == "test", "text"])) == 0, "Text appears in both train and test"

OUT.parent.mkdir(parents=True, exist_ok=True)
LOG.parent.mkdir(parents=True, exist_ok=True)

df.to_csv(OUT, index=False, encoding="utf-8")
log.add("final tsac_clean.csv", len(df), len(df), "saved with columns id,text,label,source,split")
log.save(LOG)

print("Saved:", OUT)
print("Saved:", LOG)


final tsac_clean.csv                            15185 ->   15185  (removed 0)
Saved: /home/runner/work/Data_Project/Data_Project/tunisian-sentiment/tunisian-sentiment/data/interim/tsac_clean.csv
Saved: /home/runner/work/Data_Project/Data_Project/tunisian-sentiment/tunisian-sentiment/logs/cleaning_log_tsac.csv


In [12]:
print("Final split counts:")
print(df["split"].value_counts())
print("\nFinal label counts:")
print(df["label"].value_counts())
print("\nSummary")
print({
    "train_rows": int((df["split"] == "train").sum()),
    "test_rows": int((df["split"] == "test").sum()),
    "removed_conflicting_text_count": int(removed_conflict_texts),
    "within_split_duplicate_rows_removed": int(within_split_duplicates_removed),
    "train_rows_removed_due_to_leakage": int(train_rows_removed_due_to_leakage),
    "proposal_duplicate_text_reference": 1791,
    "actual_duplicate_text_references_before_dedup": int(duplicate_text_references),
})

log.to_frame()


Final split counts:
split
train    11981
test      3204
Name: count, dtype: int64

Final label counts:
label
pos    7859
neg    7326
Name: count, dtype: int64

Summary
{'train_rows': 11981, 'test_rows': 3204, 'removed_conflicting_text_count': 12, 'within_split_duplicate_rows_removed': 1375, 'train_rows_removed_due_to_leakage': 313, 'proposal_duplicate_text_reference': 1791, 'actual_duplicate_text_references_before_dedup': 1780}


,step,rows_before,rows_after,removed,note
0,load,17069,17069,0,"TSAC train 13669, test 3400"
1,"fix_text (_x000D_, spaces)",17069,17069,0,260 rows had _x000D_; 1634 rows changed
2,remove empty rows,17069,17061,8,
3,remove rows without any letter,17061,16977,84,
4,remove label conflicts (all copies),16977,16873,104,12 conflicting texts; removed 104 rows
5,remove duplicates within split (keep first),16873,15498,1375,removed 1375 duplicate rows within split
6,remove train/test leakage (drop train copy),15498,15185,313,313 leaked texts; removed 313 train rows
7,final tsac_clean.csv,15185,15185,0,"saved with columns id,text,label,source,split"
